# 🌐 Phase 2 — Multilingual Translator: QLoRA Fine-tuning & Model Export

This notebook runs the complete Phase 2 pipeline on Kaggle with **2× NVIDIA T4 GPUs**.

## What this notebook does
1. **Environment setup** — install dependencies, authenticate with Hugging Face
2. **NLLB-200 QLoRA fine-tuning** — 4-bit quantized training with LoRA adapters
3. **Whisper-Small LoRA fine-tuning** — Indian-accent ASR across 5 languages
4. **Model export** — ONNX (fp32 + int8) and CTranslate2 formats
5. **Hub upload** — push all artifacts to Hugging Face Hub
6. **Benchmarking** — compare inference latency and file sizes

---
**Hardware**: Kaggle 2× T4 GPU · 16 GB VRAM each · 30 GB RAM  
**Base models**: `facebook/nllb-200-distilled-600M` · `openai/whisper-small`


## 1 · Environment Setup

Install all required packages. This takes 3–5 minutes on Kaggle.

In [ ]:
# Verify GPU availability
import subprocess, sys
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"GPU count       : {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB")

In [ ]:
%%capture
!pip install -q -r /kaggle/input/phase2-code/requirements.txt

# bitsandbytes often needs a CUDA-compatible build; use the official release
!pip install -q bitsandbytes==0.43.0 --extra-index-url https://download.pytorch.org/whl/cu118

# CTranslate2 (CPU + CUDA)
!pip install -q ctranslate2==4.1.0

print('Installation complete.')

In [ ]:
# Core imports — confirm versions
import transformers, peft, datasets, evaluate, optimum, bitsandbytes
import onnxruntime, ctranslate2

print(f"transformers : {transformers.__version__}")
print(f"peft         : {peft.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"evaluate     : {evaluate.__version__}")
print(f"optimum      : {optimum.__version__}")
print(f"bitsandbytes : {bitsandbytes.__version__}")
print(f"onnxruntime  : {onnxruntime.__version__}")
print(f"ctranslate2  : {ctranslate2.__version__}")

## 2 · Hugging Face Authentication

Add your `HF_TOKEN` and `HF_USERNAME` as **Kaggle Secrets** (Settings → Add-ons → Secrets)  
or paste them directly below (never commit secrets to version control).

In [ ]:
import os
from huggingface_hub import login

# ── Option A: Kaggle Secrets ─────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    HF_TOKEN    = secrets.get_secret("HF_TOKEN")
    HF_USERNAME = secrets.get_secret("HF_USERNAME")
    print("Loaded credentials from Kaggle Secrets.")
except Exception:
    # ── Option B: Environment variables or manual entry ───────────────────────
    HF_TOKEN    = os.environ.get("HF_TOKEN", "")     # or paste token here
    HF_USERNAME = os.environ.get("HF_USERNAME", "")  # or paste username here
    print("Loaded credentials from environment variables.")

if HF_TOKEN:
    login(token=HF_TOKEN)
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HF_USERNAME"] = HF_USERNAME
    print(f"Logged in as: {HF_USERNAME}")
else:
    print("⚠ No HF_TOKEN found — Hub push/pull will be disabled.")

## 3 · Load Phase 2 Scripts

Copy the Phase 2 Python scripts from the Kaggle dataset input into the working directory.

In [ ]:
import shutil
from pathlib import Path

# Scripts are uploaded as a Kaggle Dataset at /kaggle/input/phase2-code/
# If running inline, they are already available in the working dir.
script_src = Path("/kaggle/input/phase2-code")
script_dst = Path("/kaggle/working")

for script in ["finetune_nllb.py", "finetune_whisper.py", "export_models.py"]:
    src = script_src / script
    if src.exists():
        shutil.copy(src, script_dst / script)
        print(f"Copied: {script}")
    else:
        print(f"⚠ Not found in input dataset: {script} (will use inline version)")

print("\nWorking directory contents:")
for f in sorted(script_dst.iterdir()):
    print(f"  {f.name}")

## 4 · NLLB-200-distilled-600M QLoRA Fine-tuning

### Architecture overview

```
facebook/nllb-200-distilled-600M   (frozen, 4-bit NF4)
         │
         ▼
  LoRA adapters (r=16, α=32)
  Targets: q_proj, v_proj, k_proj, out_proj, fc1, fc2
  Trainable params: ~12M / 600M total
         │
         ▼
  Seq2SeqTrainer (3 epochs, BLEU metric)
         │
         ▼
  Hub push: {username}/nllb-200-600m-qlora
```

**Expected training time**: ~2 hours on 2× T4 (en→hi, opus-100 fallback, 500K samples)

In [ ]:
# ── Inline NLLB fine-tuning (uses finetune_nllb.py logic) ────────────────────
import sys
sys.path.insert(0, '/kaggle/working')

# Patch argv for argparse
sys.argv = [
    'finetune_nllb.py',
    '--hf_username',   HF_USERNAME,
    '--hf_token',      HF_TOKEN,
    '--output_dir',    '/kaggle/working/nllb_qlora',
    '--merged_dir',    '/kaggle/working/nllb_merged',
    '--onnx_dir',      '/kaggle/working/nllb_onnx',
    '--ct2_dir',       '/kaggle/working/nllb_ct2',
    '--num_train_epochs', '3',
    '--per_device_train_batch_size', '8',
    '--gradient_accumulation_steps', '4',
    '--learning_rate', '2e-4',
    '--export_onnx',
    '--export_ct2',
    '--push_to_hub',
]

print('Starting NLLB QLoRA fine-tuning …')
print('This will take approximately 2 hours on 2× T4 GPUs.')

In [ ]:
# Run NLLB fine-tuning
# Import and call main() directly for better Kaggle notebook integration
import importlib, finetune_nllb
importlib.reload(finetune_nllb)  # ensure fresh module state

finetune_nllb.main()

In [ ]:
# Verify NLLB outputs
import os
from pathlib import Path

def show_dir(path, label):
    p = Path(path)
    if p.exists():
        files = list(p.rglob('*'))
        total_size = sum(f.stat().st_size for f in files if f.is_file()) / 1e6
        print(f"\n{label}: {total_size:.1f} MB ({len(files)} files)")
        for f in sorted(files)[:20]:
            if f.is_file():
                print(f"  {f.relative_to(p)}: {f.stat().st_size / 1e6:.2f} MB")
    else:
        print(f"\n{label}: NOT FOUND")

show_dir('/kaggle/working/nllb_qlora',   'NLLB Adapter (LoRA)')
show_dir('/kaggle/working/nllb_merged',  'NLLB Merged (Full)')
show_dir('/kaggle/working/nllb_onnx',    'NLLB ONNX')
show_dir('/kaggle/working/nllb_ct2',     'NLLB CTranslate2')

## 5 · Whisper-Small LoRA Fine-tuning (Indian Accent ASR)

### Dataset breakdown

| Source | Languages | Approx. samples |
|---|---|---|
| Common Voice 13 | hi, ta, bn, te, mr | ~25,000 (5K × 5) |
| FLEURS | hi_in, ta_in, bn_in, te_in, mr_in | ~10,000 (2K × 5) |
| **Total** | **5 languages** | **~35,000** |

### LoRA config
- Target modules: `q_proj`, `v_proj` (encoder + decoder attention)
- r=32, α=64 → ~6M trainable parameters

**Expected training time**: ~3 hours on 2× T4

In [ ]:
import gc, torch

# Free GPU memory from NLLB training before Whisper training
gc.collect()
torch.cuda.empty_cache()

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        mem = torch.cuda.memory_reserved(i) / 1e9
        print(f"GPU {i} reserved memory: {mem:.2f} GB")

print('Memory cleared. Ready for Whisper training.')

In [ ]:
# Patch argv for Whisper fine-tuning
sys.argv = [
    'finetune_whisper.py',
    '--hf_username',    HF_USERNAME,
    '--hf_token',       HF_TOKEN,
    '--output_dir',     '/kaggle/working/whisper_lora',
    '--merged_dir',     '/kaggle/working/whisper_merged',
    '--onnx_dir',       '/kaggle/working/whisper_onnx',
    '--num_train_epochs', '5',
    '--per_device_train_batch_size', '16',
    '--gradient_accumulation_steps', '2',
    '--learning_rate', '1e-4',
    '--warmup_steps', '500',
    '--max_cv_samples_per_lang', '5000',
    '--use_fleurs',
    '--export_onnx',
    '--push_to_hub',
]

print('Starting Whisper LoRA fine-tuning …')
print('This will take approximately 3 hours on 2× T4 GPUs.')

In [ ]:
import finetune_whisper
importlib.reload(finetune_whisper)

finetune_whisper.main()

In [ ]:
# Verify Whisper outputs
show_dir('/kaggle/working/whisper_lora',    'Whisper Adapter (LoRA)')
show_dir('/kaggle/working/whisper_merged',  'Whisper Merged (Full)')
show_dir('/kaggle/working/whisper_onnx',    'Whisper ONNX')

## 6 · Standalone Export & Benchmarking

This section runs the standalone `export_models.py` script which:
- Re-exports everything to ONNX (fp32) and quantized (int8)
- Converts NLLB to CTranslate2 (int8)
- Benchmarks CPU inference latency
- Reports final file sizes
- Pushes everything to the Hub

In [ ]:
gc.collect()
torch.cuda.empty_cache()

sys.argv = [
    'export_models.py',
    '--nllb_model_dir',    '/kaggle/working/nllb_merged',
    '--whisper_model_dir', '/kaggle/working/whisper_merged',
    '--output_base',       '/kaggle/working/exports',
    '--hf_username',       HF_USERNAME,
    '--hf_token',          HF_TOKEN,
    '--export_nllb',
    '--export_whisper',
    '--quantize_onnx',
    '--export_ct2',
    '--push_to_hub',
    '--run_benchmark',
]

print('Running export pipeline …')

In [ ]:
import export_models
importlib.reload(export_models)

export_models.main()

## 7 · End-to-End Inference Demo

Quick sanity check: translate an English sentence to Hindi using the fine-tuned model.

In [ ]:
# NLLB CTranslate2 inference demo
import ctranslate2
from transformers import NllbTokenizer

CT2_DIR = '/kaggle/working/nllb_ct2'

try:
    translator = ctranslate2.Translator(CT2_DIR, device='cpu', inter_threads=4)
    tokenizer  = NllbTokenizer.from_pretrained('facebook/nllb-200-distilled-600M')
    tokenizer.src_lang = 'eng_Latn'

    test_sentences = [
        "Hello, how are you?",
        "The weather is very nice today.",
        "Please show me the way to the nearest hospital.",
        "What is the price of this item?",
    ]

    print("English → Hindi (CTranslate2)\n" + "─" * 60)
    for text in test_sentences:
        tokens = tokenizer.convert_ids_to_tokens(
            tokenizer.encode(text, add_special_tokens=False)
        )
        result = translator.translate_batch(
            [tokens],
            target_prefix=[['hin_Deva']],
            max_decoding_length=200,
            beam_size=4,
        )
        output_tokens = result[0].hypotheses[0][1:]  # remove lang token
        output_text   = tokenizer.decode(
            tokenizer.convert_tokens_to_ids(output_tokens),
            skip_special_tokens=True
        )
        print(f"EN: {text}")
        print(f"HI: {output_text}")
        print()

except Exception as e:
    print(f'CTranslate2 demo error: {e}')
    print('Falling back to Transformers pipeline...')

    from transformers import pipeline
    translator_hf = pipeline(
        'translation',
        model='/kaggle/working/nllb_merged',
        tokenizer='/kaggle/working/nllb_merged',
        src_lang='eng_Latn',
        tgt_lang='hin_Deva',
        device=0 if torch.cuda.is_available() else -1,
    )
    for text in test_sentences:
        result = translator_hf(text, max_length=200)
        print(f"EN: {text}")
        print(f"HI: {result[0]['translation_text']}")
        print()

In [ ]:
# Whisper ONNX inference demo with a synthetic audio signal
import numpy as np
from transformers import WhisperProcessor, WhisperForConditionalGeneration

print('Whisper merged model — ASR demo with synthetic audio …')
print('(In production, provide a real .wav file)')

try:
    processor = WhisperProcessor.from_pretrained('/kaggle/working/whisper_merged')
    model_wh  = WhisperForConditionalGeneration.from_pretrained(
        '/kaggle/working/whisper_merged',
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map='auto',
    )
    model_wh.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
        language='hi', task='transcribe'
    )

    # Synthetic 3-second audio at 16 kHz (replace with real audio for actual use)
    dummy_audio = np.zeros(16000 * 3, dtype=np.float32)
    input_features = processor(
        dummy_audio,
        sampling_rate=16000,
        return_tensors='pt',
    ).input_features

    with torch.no_grad():
        predicted_ids = model_wh.generate(input_features.to(model_wh.device))
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
    print(f'Transcription: {transcription[0]!r} (expected empty for silent audio)')

except Exception as e:
    print(f'Whisper demo error: {e}')

## 8 · Final Summary & File Sizes

In [ ]:
import os
from pathlib import Path

print('\n' + '=' * 70)
print('PHASE 2 — FINAL SUMMARY')
print('=' * 70)

artifacts = {
    'NLLB Adapter (LoRA)':          '/kaggle/working/nllb_qlora',
    'NLLB Merged (fp16)':           '/kaggle/working/nllb_merged',
    'NLLB ONNX (fp32)':             '/kaggle/working/nllb_onnx',
    'NLLB ONNX (int8)':             '/kaggle/working/exports/nllb_onnx_int8',
    'NLLB CTranslate2 (int8)':      '/kaggle/working/nllb_ct2',
    'Whisper Adapter (LoRA)':       '/kaggle/working/whisper_lora',
    'Whisper Merged (fp16)':        '/kaggle/working/whisper_merged',
    'Whisper ONNX (fp32)':          '/kaggle/working/whisper_onnx',
    'Whisper ONNX (int8)':          '/kaggle/working/exports/whisper_onnx_int8',
}

total_size = 0
print(f'\n{"Artifact":<35} {"Size (MB)":>12} {"Status"}')
print('-' * 65)

for label, path in artifacts.items():
    p = Path(path)
    if p.exists():
        size = sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) / 1e6
        total_size += size
        status = '✓'
    else:
        size = 0
        status = '✗ MISSING'
    print(f'{label:<35} {size:>10.1f} MB  {status}')

print('-' * 65)
print(f'{"TOTAL":<35} {total_size:>10.1f} MB')

print('\n' + '=' * 70)
print(f'Hub repos:')
print(f'  NLLB  : https://huggingface.co/{HF_USERNAME}/nllb-200-600m-qlora')
print(f'  Whisper: https://huggingface.co/{HF_USERNAME}/whisper-small-indian-lora')
print('=' * 70)

In [ ]:
# Training metrics visualization
import json
import matplotlib.pyplot as plt
from pathlib import Path

def plot_training_metrics(trainer_state_path, title, metric_name, ax):
    """Plot training and eval metrics from trainer_state.json."""
    p = Path(trainer_state_path)
    if not p.exists():
        ax.text(0.5, 0.5, 'No metrics data', ha='center', va='center')
        ax.set_title(title)
        return

    with open(p) as f:
        state = json.load(f)

    history = state.get('log_history', [])
    train_steps, train_vals = [], []
    eval_steps,  eval_vals  = [], []

    for entry in history:
        if f'eval_{metric_name}' in entry:
            eval_steps.append(entry['step'])
            eval_vals.append(entry[f'eval_{metric_name}'])
        elif 'loss' in entry and metric_name == 'loss':
            train_steps.append(entry['step'])
            train_vals.append(entry['loss'])

    if train_vals:
        ax.plot(train_steps, train_vals, label='Train loss', alpha=0.7)
    if eval_vals:
        ax.plot(eval_steps, eval_vals, label=f'Eval {metric_name}', marker='o')

    ax.set_title(title)
    ax.set_xlabel('Step')
    ax.set_ylabel(metric_name.upper())
    ax.legend()
    ax.grid(True, alpha=0.3)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Phase 2 Training Metrics', fontsize=14, fontweight='bold')

plot_training_metrics(
    '/kaggle/working/nllb_qlora/trainer_state.json',
    'NLLB QLoRA — BLEU Score',
    'bleu', axes[0]
)
plot_training_metrics(
    '/kaggle/working/whisper_lora/trainer_state.json',
    'Whisper LoRA — WER',
    'wer', axes[1]
)

plt.tight_layout()
plt.savefig('/kaggle/working/training_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training metrics plot saved to: /kaggle/working/training_metrics.png')

## 9 · Next Steps (Phase 3)

Phase 2 is complete! Here's what Phase 3 will build on top of these artifacts:

| Component | Phase 3 Plan |
|---|---|
| **API Backend** | FastAPI service wrapping NLLB (CTranslate2) + Whisper (ONNX) |
| **TTS Integration** | Add Coqui TTS or VITS for text-to-speech output |
| **Web UI** | Gradio / Streamlit app with audio recording + translation |
| **Mobile** | Export Whisper to TFLite for on-device Android ASR |
| **Evaluation** | FLORES-200 benchmark across all 11 language pairs |

### Hub artifacts from this phase
- 🔗 [`{username}/nllb-200-600m-qlora`](https://huggingface.co/{username}/nllb-200-600m-qlora) — NLLB adapters, merged model, ONNX, CTranslate2
- 🔗 [`{username}/whisper-small-indian-lora`](https://huggingface.co/{username}/whisper-small-indian-lora) — Whisper adapters, merged model, ONNX